In [5]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_keras_model_accurately(model_path, model_name):
    print(f"\n🔷 Model: {model_name}")
    
    # Load model
    model = load_model(model_path)
    input_shape = model.input_shape[1:3]

    # Correct test preprocessing to match training
    datagen = ImageDataGenerator(rescale=1./255)

    test_generator = datagen.flow_from_directory(
        'D:/college/grad project/integrated/testing',
        target_size=input_shape,
        batch_size=32,
        class_mode='categorical',
        shuffle=False
    )

    # Predict
    y_true = test_generator.classes
    y_prob = model.predict(test_generator)
    y_pred = np.argmax(y_prob, axis=1)

    # Classification report
    print(f"\n🔷 Model: {model_name}")
    class_names = list(test_generator.class_indices.keys())
    print("\n📊 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))

    # Confusion matrix and per-class accuracy
    cm = confusion_matrix(y_true, y_pred)
    print("\n📌 Per-Class Accuracy:")
    per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
    for i, class_name in enumerate(class_names):
        print(f"{class_name}: {per_class_accuracy[i]*100:.2f}%")

    print("\n📌 Confusion Matrix:")
    print(cm)

In [25]:
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import timm
import torch.nn as nn
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def evaluate_vit_model(model_path, model_name):
    print(f"\n🔷 Model: {model_name}")

    # ✅ ViT requires ImageNet normalization
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5,), std=(0.5,))
    ])

    # Load test data
    test_dataset = ImageFolder('D:/college/grad project/integrated/testing', transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    class_names = test_dataset.classes
    num_classes = len(class_names)

    # Load model
    model = timm.create_model('vit_base_patch16_224', pretrained=False)
    model.head = nn.Linear(model.head.in_features, num_classes)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()

    # Predict
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Classification report
    print(f"\n🔷 Model: {model_name}")
    print("\n📊 Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    print("\n📌 Per-Class Recall (True Positive Rate):")
    per_class_recall = cm.diagonal() / cm.sum(axis=1)
    for i, class_name in enumerate(class_names):
        print(f"{class_name}: {per_class_recall[i]*100:.2f}%")

    print("\n📌 Confusion Matrix:")
    print(cm)

In [7]:
evaluate_keras_model_accurately("saved_model/mri_classifier_final.keras", "ResNet50 Final")


🔷 Model: ResNet50 Final
Found 1705 images belonging to 4 classes.


D:\anaconda\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


54/54 ━━━━━━━━━━━━━━━━━━━━ 29s 508ms/step

🔷 Model: ResNet50 Final

📊 Classification Report:
                  precision    recall  f1-score   support

    glioma_tumor       0.97      0.85      0.91       400
meningioma_tumor       0.92      0.94      0.93       421
        no_tumor       0.91      1.00      0.95       510
 pituitary_tumor       0.97      0.95      0.96       374

        accuracy                           0.94      1705
       macro avg       0.94      0.93      0.94      1705
    weighted avg       0.94      0.94      0.94      1705


📌 Per-Class Accuracy:
glioma_tumor: 85.00%
meningioma_tumor: 93.59%
no_tumor: 100.00%
pituitary_tumor: 94.92%

📌 Confusion Matrix:
[[340  25  34   1]
 [ 10 394   6  11]
 [  0   0 510   0]
 [  1   8  10 355]]


In [11]:
evaluate_keras_model_accurately("efficientnet_b3_finetuned_v2.keras", "Efficientnetb3")


🔷 Model: EfficientnetB3
Found 1705 images belonging to 4 classes.


D:\anaconda\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


54/54 ━━━━━━━━━━━━━━━━━━━━ 73s 1s/step

🔷 Model: EfficientnetB3

📊 Classification Report:
                  precision    recall  f1-score   support

    glioma_tumor       0.91      0.90      0.90       400
meningioma_tumor       0.93      0.93      0.93       421
        no_tumor       0.96      0.98      0.97       510
 pituitary_tumor       0.98      0.97      0.97       374

        accuracy                           0.94      1705
       macro avg       0.94      0.94      0.94      1705
    weighted avg       0.94      0.94      0.94      1705


📌 Per-Class Accuracy:
glioma_tumor: 89.50%
meningioma_tumor: 92.64%
no_tumor: 97.65%
pituitary_tumor: 97.33%

📌 Confusion Matrix:
[[358  23  16   3]
 [ 21 390   4   6]
 [ 10   2 498   0]
 [  4   5   1 364]]


In [15]:
evaluate_keras_model_accurately("saved_model/inceptionresnetv2_mri_final.keras", "InceptionResnetV2")


🔷 Model: InceptionResnetV2
Found 1705 images belonging to 4 classes.
54/54 ━━━━━━━━━━━━━━━━━━━━ 117s 2s/step

🔷 Model: InceptionResnetV2

📊 Classification Report:
                  precision    recall  f1-score   support

    glioma_tumor       1.00      0.90      0.95       400
meningioma_tumor       0.93      1.00      0.96       421
        no_tumor       0.96      1.00      0.98       510
 pituitary_tumor       1.00      0.97      0.98       374

        accuracy                           0.97      1705
       macro avg       0.97      0.97      0.97      1705
    weighted avg       0.97      0.97      0.97      1705


📌 Per-Class Accuracy:
glioma_tumor: 89.75%
meningioma_tumor: 99.76%
no_tumor: 100.00%
pituitary_tumor: 96.79%

📌 Confusion Matrix:
[[359  27  14   0]
 [  0 420   0   1]
 [  0   0 510   0]
 [  0   6   6 362]]


In [27]:
evaluate_vit_model("saved_model/Vitbase_model.pth", "ViT Base")


🔷 Model: ViT Base

🔷 Model: ViT Base

📊 Classification Report:
                  precision    recall  f1-score   support

    glioma_tumor       0.97      0.90      0.93       400
meningioma_tumor       0.94      0.97      0.95       421
        no_tumor       0.97      1.00      0.98       510
 pituitary_tumor       0.97      0.97      0.97       374

        accuracy                           0.96      1705
       macro avg       0.96      0.96      0.96      1705
    weighted avg       0.96      0.96      0.96      1705


📌 Per-Class Recall (True Positive Rate):
glioma_tumor: 89.75%
meningioma_tumor: 96.67%
no_tumor: 100.00%
pituitary_tumor: 97.33%

📌 Confusion Matrix:
[[359  21  17   3]
 [  7 407   0   7]
 [  0   0 510   0]
 [  4   6   0 364]]
